# Supervised Learning Models

## Basic Regression Models


| Situation              | Best choice         |
| ---------------------- | ------------------- |
| Small data + inference | OLS                 |
| Large data             | Gradient Descent    |
| Multicollinearity      | Ridge               |
| Feature selection      | Lasso               |
| Correlated + selection | Elastic Net         |
| Visualization / EDA    | regplot             |
| Binary target          | Logistic Regression |


| Model            | CV Method         |
| ---------------- | ----------------- |
| OLS              | `cross_val_score` |
| Gradient Descent | `cross_val_score` |
| Ridge            | `RidgeCV`    |
| Lasso            | `LassoCV`         |
| Elastic Net      | `ElasticNetCV`    |
| Logistic         | `GridSearchCV` |
| Time series      | `TimeSeriesSplit` |


## Linear Regression Model (Simple or Multiple)

In [ ]:
def linear_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,
    random_state=42
):
    """
    Linear Regression with:
    - k-Fold Cross-Validation ONLY on training data
    - Performance evaluation on BOTH train and test sets
    - Proper preprocessing using Pipeline (no data leakage)
    - Diagnostic plots
    - Returns final trained model

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training features
    y_train : array-like or Series
        Training target
    X_test : array-like or DataFrame
        Test features
    y_test : array-like or Series
        Test target
    cv : int, default=5
        Number of CV folds (applied only on training data)
    scale : bool, default=True
        Whether to apply StandardScaler
    random_state : int, default=42
        For reproducibility

    Returns
    -------
    model : sklearn Pipeline
        Final trained Linear Regression model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold, cross_validate
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Build Pipeline (Prevents Data Leakage)
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", LinearRegression()))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Cross-Validation ONLY on Training Data
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    # Convert negative metrics
    cv_mse = -cv_results["test_mse"]
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]
    cv_rmse = np.sqrt(cv_mse)

    # -------------------------------------------------------
    # 3. Print Cross-Validation Results (TRAIN ONLY)
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")

    for i in range(cv):
        print(f"Fold {i+1}: "
              f"MSE={cv_mse[i]:.4f}, "
              f"RMSE={cv_rmse[i]:.4f}, "
              f"MAE={cv_mae[i]:.4f}, "
              f"R2={cv_r2[i]:.4f}")

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 4. CV Error Distribution Plot
    # -------------------------------------------------------
    plt.figure()
    plt.boxplot(cv_mse)
    plt.title("Cross-Validation MSE Distribution (Train Data)")
    plt.ylabel("MSE")
    plt.show()

    # -------------------------------------------------------
    # 5. Train Final Model on FULL TRAIN DATA
    # -------------------------------------------------------
    pipeline.fit(X_train, y_train)

    # Predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # -------------------------------------------------------
    # 6. Performance Metrics (TRAIN DATA)
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print("Coefficient or slope:", pipeline.named_steps["model"].coef_)
    print("Intercept:", pipeline.named_steps["model"].intercept_)
    
    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")
    print(f"Adjusted R2   : {(1-(1-train_r2)*(len(y_train)-1)/(len(y_train)-X_train.shape[1]-1)):.4f}")

    # -------------------------------------------------------
    # 7. Performance Metrics (TEST DATA)
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")
    print(f"Adjusted R2   : {(1-(1-test_r2)*(len(y_test)-1)/(len(y_test)-X_test.shape[1]-1)):.4f}")
    
    
    # -------------------------------------------------------
    # 8. Actual vs Predicted (TRAIN DATA)
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_train, y_train_pred)
    plt.plot([y_train.min(), y_train.max()],
             [y_train.min(), y_train.max()])
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Actual vs Predicted (Train Data)")
    plt.show()
    

    # -------------------------------------------------------
    # 9. Actual vs Predicted (TEST DATA)
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Actual vs Predicted (Test Data)")
    plt.show()

    # -------------------------------------------------------
    # 10. Residual Plot (TEST DATA)
    # -------------------------------------------------------
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test Data)")
    plt.show()
    
    
    # -------------------------------------------------------
    # 11. Residual Plot (TEST DATA)
    # -------------------------------------------------------   
    sns.displot(residuals,kind='kde')
    
    # -------------------------------------------------------
    # 12. Return Final Model
    # -------------------------------------------------------
    return pipeline


## Linear Regression (OLS Method)

In [ ]:
def linear_regression_ols(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True
):
    """
    OLS Regression using statsmodels with:
    - k-Fold Cross-Validation ONLY on training data
    - Full statistical summary (OLS)
    - Performance metrics on train & test
    - Diagnostic plots
    - Returns final fitted OLS model

    Parameters
    ----------
    X_train, y_train : training data
    X_test, y_test   : test data
    cv               : number of CV folds
    scale            : whether to standardize features

    Returns
    -------
    model : statsmodels OLS fitted model
    """

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import statsmodels.api as sm

    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Feature Scaling (TRAIN ONLY)
    # -------------------------------------------------------
    if scale:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_scaled = X_train
        X_test_scaled = X_test

    # -------------------------------------------------------
    # 2. Add Intercept Term (MANDATORY for OLS)
    # -------------------------------------------------------
    X_train_const = sm.add_constant(X_train_scaled)
    X_test_const = sm.add_constant(X_test_scaled)

    # -------------------------------------------------------
    # 3. k-Fold Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)

    cv_mse, cv_mae, cv_r2 = [], [], []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_const), 1):
        X_tr, X_val = X_train_const[train_idx], X_train_const[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = sm.OLS(y_tr, X_tr).fit()
        y_val_pred = model.predict(X_val)

        cv_mse.append(mean_squared_error(y_val, y_val_pred))
        cv_mae.append(mean_absolute_error(y_val, y_val_pred))
        cv_r2.append(r2_score(y_val, y_val_pred))

        print(f"Fold {fold}: "
              f"MSE={cv_mse[-1]:.4f}, "
              f"MAE={cv_mae[-1]:.4f}, "
              f"R2={cv_r2[-1]:.4f}")

    print("\n========== MEAN CV PERFORMANCE (TRAIN DATA) ==========")
    print(f"Mean MSE  : {np.mean(cv_mse):.4f}")
    print(f"Mean MAE  : {np.mean(cv_mae):.4f}")
    print(f"Mean R2   : {np.mean(cv_r2):.4f}")

    # -------------------------------------------------------
    # 4. CV Error Distribution Plot
    # -------------------------------------------------------
    plt.figure()
    plt.boxplot(cv_mse)
    plt.title("CV MSE Distribution (OLS – Train Data)")
    plt.ylabel("MSE")
    plt.show()

    # -------------------------------------------------------
    # 5. Train FINAL OLS Model on FULL TRAIN DATA
    # -------------------------------------------------------
    final_model = sm.OLS(y_train, X_train_const).fit()

    print("\n========== OLS STATISTICAL SUMMARY ==========\n")
    print(final_model.summary())

    # -------------------------------------------------------
    # 6. Predictions
    # -------------------------------------------------------
    y_train_pred = final_model.predict(X_train_const)
    y_test_pred = final_model.predict(X_test_const)

    # -------------------------------------------------------
    # 7. Train Metrics
    # -------------------------------------------------------
    print("\n========== TRAIN PERFORMANCE ==========")
    print(f"MSE  : {mean_squared_error(y_train, y_train_pred):.4f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
    print(f"MAE  : {mean_absolute_error(y_train, y_train_pred):.4f}")
    print(f"R2   : {r2_score(y_train, y_train_pred):.4f}")

    # -------------------------------------------------------
    # 8. Test Metrics
    # -------------------------------------------------------
    print("\n========== TEST PERFORMANCE ==========")
    print(f"MSE  : {mean_squared_error(y_test, y_test_pred):.4f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
    print(f"MAE  : {mean_absolute_error(y_test, y_test_pred):.4f}")
    print(f"R2   : {r2_score(y_test, y_test_pred):.4f}")

    # -------------------------------------------------------
    # 9. Actual vs Predicted (TEST)
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (OLS – Test Data)")
    plt.show()

    # -------------------------------------------------------
    # 10. Residual Plot (TEST)
    # -------------------------------------------------------
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (OLS – Test Data)")
    plt.show()

    return final_model


## Polynomial Regression

In [ ]:
def polynomial_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    degrees,
    cv=5,
    scale=True,
    random_state=42
):
    """
    Polynomial Regression with:
    - Degree selection using k-Fold Cross-Validation (TRAIN data only)
    - Proper preprocessing using Pipeline (no data leakage)
    - Performance evaluation on TRAIN and TEST sets
    - Diagnostic plots
    - Returns final trained model (best degree)

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    degrees : list or range
        Polynomial degrees to try (e.g. [1,2,3,4] or range(1,6))
    cv : int, default=5
        Number of CV folds
    scale : bool, default=True
        Whether to apply StandardScaler
    random_state : int, default=42

    Returns
    -------
    best_pipeline : sklearn Pipeline
        Final trained Polynomial Regression model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.preprocessing import PolynomialFeatures, StandardScaler
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import KFold, cross_validate
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # -------------------------------------------------------
    # 1. CV setup
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    degree_results = {}

    # -------------------------------------------------------
    # 2. Cross-validation for EACH degree (TRAIN ONLY)
    # -------------------------------------------------------
    print("\n========== DEGREE SELECTION USING CV (TRAIN DATA ONLY) ==========\n")

    for deg in degrees:
        steps = [
            ("poly", PolynomialFeatures(degree=deg, include_bias=False))
        ]

        if scale:
            steps.append(("scaler", StandardScaler()))

        steps.append(("model", LinearRegression()))

        pipeline = Pipeline(steps)

        cv_results = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=kf,
            scoring=scoring,
            return_train_score=True
        )

        mse = -cv_results["test_mse"]
        rmse = np.sqrt(mse)
        mae = -cv_results["test_mae"]
        r2 = cv_results["test_r2"]

        degree_results[deg] = {
            "mse": mse.mean(),
            "rmse": rmse.mean(),
            "mae": mae.mean(),
            "r2": r2.mean()
        }

        print(f"Degree {deg}: "
              f"MSE={mse.mean():.4f}, "
              f"RMSE={rmse.mean():.4f}, "
              f"MAE={mae.mean():.4f}, "
              f"R2={r2.mean():.4f}")

    # -------------------------------------------------------
    # 3. Select BEST degree (highest CV R²)
    # -------------------------------------------------------
    best_degree = max(degree_results, key=lambda d: degree_results[d]["r2"])

    print("\n---------- BEST DEGREE SELECTED ----------")
    print(f"Best Degree: {best_degree}")
    print(f"Best CV R2 : {degree_results[best_degree]['r2']:.4f}")

    # -------------------------------------------------------
    # 4. Train FINAL model on FULL TRAIN data
    # -------------------------------------------------------
    final_steps = [
        ("poly", PolynomialFeatures(degree=best_degree, include_bias=False))
    ]

    if scale:
        final_steps.append(("scaler", StandardScaler()))

    final_steps.append(("model", LinearRegression()))

    best_pipeline = Pipeline(final_steps)
    best_pipeline.fit(X_train, y_train)

    # Predictions
    y_train_pred = best_pipeline.predict(X_train)
    y_test_pred = best_pipeline.predict(X_test)

    # -------------------------------------------------------
    # 5. TRAIN metrics
    # -------------------------------------------------------
    print("\n=========== TRAIN DATA PERFORMANCE ==========")

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")

    # -------------------------------------------------------
    # 6. TEST metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")

    # -------------------------------------------------------
    # 7. Actual vs Predicted plots
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_train, y_train_pred)
    plt.plot([y_train.min(), y_train.max()],
             [y_train.min(), y_train.max()])
    plt.title("Actual vs Predicted (Train)")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.show()

    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.title("Actual vs Predicted (Test)")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.show()

    # --------------------------------------------------------
    # 8. Residual analysis (TEST)
    # --------------------------------------------------------
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 9. Return final model
    # -------------------------------------------------------
    return best_pipeline


## Ridge Regression [Linear]

In [ ]:
def ridge_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    model_type="ridge",       # "ridge" or "ridgecv"
    alpha=None,               # used for Ridge
    alphas=None,              # used for RidgeCV
    cv=5,
    scale=True,
    random_state=42
):
    """
    Ridge Regression / RidgeCV with:
    - k-Fold Cross-Validation ONLY on training data
    - Performance evaluation on BOTH train and test sets
    - Proper preprocessing using Pipeline (no data leakage)
    - Diagnostic plots
    - Flexible model choice (Ridge or RidgeCV)

    Parameters
    ----------
    X_train, y_train : training data
    X_test, y_test   : test data
    model_type : {"ridge", "ridgecv"}
        Choose Ridge or RidgeCV
    alpha : float, optional
        Regularization strength for Ridge
    alphas : list or array, optional
        Candidate alphas for RidgeCV
    cv : int, default=5
        Number of CV folds
    scale : bool, default=True
        Whether to apply StandardScaler
    random_state : int, default=42

    Returns
    -------
    model : sklearn Pipeline
        Final trained Ridge / RidgeCV model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import Ridge, RidgeCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold, cross_validate
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Handle Defaults
    # -------------------------------------------------------
    if model_type.lower() == "ridge":
        if alpha is None:
            alpha = 1.0   # default Ridge alpha
        model = Ridge(alpha=alpha, random_state=random_state)

    elif model_type.lower() == "ridgecv":
        if alphas is None:
            alphas = np.logspace(-3, 3, 50)  # default alpha grid
        model = RidgeCV(alphas=alphas, cv=cv)

    else:
        raise ValueError("model_type must be 'ridge' or 'ridgecv'")

    # -------------------------------------------------------
    # 2. Build Pipeline (No Data Leakage)
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 3. Cross-Validation ONLY on Training Data
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]
    cv_rmse = np.sqrt(cv_mse)

    # -------------------------------------------------------
    # 4. Print CV Results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")

    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. CV Error Distribution Plot
    # -------------------------------------------------------
    plt.figure()
    plt.boxplot(cv_mse)
    plt.title("Cross-Validation MSE Distribution (Train Data)")
    plt.ylabel("MSE")
    plt.show()

    # -------------------------------------------------------
    # 6. Train Final Model on FULL TRAIN DATA
    # -------------------------------------------------------
    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # -------------------------------------------------------
    # 7. Train Performance
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print("Coefficients:", pipeline.named_steps["model"].coef_)
    print("Intercept:", pipeline.named_steps["model"].intercept_)

    if model_type.lower() == "ridgecv":
        print("Best Alpha (RidgeCV):", pipeline.named_steps["model"].alpha_)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - train_r2) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 8. Test Performance
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - test_r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 9. Diagnostic Plots
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test Data)")
    plt.show()

    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test Data)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 10. Return Final Model
    # -------------------------------------------------------
    return pipeline


## Lasso Regression [Linear]

In [ ]:
def lasso_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    model_type="lasso",      # "lasso" or "lassocv"
    alpha=None,              # for Lasso
    alphas=None,             # for LassoCV
    cv=5,
    scale=True,
    random_state=42,
    max_iter=5000
):
    """
    Lasso / LassoCV Regression with:
    - k-Fold CV only on training data
    - Proper pipeline (no leakage)
    - Feature selection via L1 regularization
    - Full diagnostics and metrics
    
        Parameters
        ----------
        X_train, y_train : training data
        X_test, y_test   : test data
        model_type : {"lasso", "lassocv"}
        alpha : float, optional
            Regularization strength for Lasso
        alphas : list or array, optional
            Candidate alphas for LassoCV
        cv : int, default=5
            Number of CV folds
        scale : bool, default=True
            Whether to apply StandardScaler
        random_state : int, default=42
        max_iter: 5000

        Returns
        -------
        model : sklearn Pipeline
            Final trained Lasso / LassoCV model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import Lasso, LassoCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold, cross_validate
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # -------------------------------------------------------
    # 1. Handle Defaults
    # -------------------------------------------------------
    if model_type.lower() == "lasso":
        if alpha is None:
            alpha = 0.01
        model = Lasso(alpha=alpha, max_iter=max_iter, random_state=random_state)

    elif model_type.lower() == "lassocv":
        if alphas is None:
            alphas = np.logspace(-4, 1, 50)
        model = LassoCV(alphas=alphas, cv=cv, max_iter=max_iter, random_state=random_state)

    else:
        raise ValueError("model_type must be 'lasso' or 'lassocv'")

    # -------------------------------------------------------
    # 2. Pipeline
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 3. Cross-Validation ONLY on Training Data
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]
    cv_rmse = np.sqrt(cv_mse)

    # -------------------------------------------------------
    # 4. Print CV Results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")

    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. CV Error Distribution Plot
    # -------------------------------------------------------
    plt.figure()
    plt.boxplot(cv_mse)
    plt.title("Cross-Validation MSE Distribution (Train Data)")
    plt.ylabel("MSE")
    plt.show()

    # -------------------------------------------------------
    # 6. Train Final Model on FULL TRAIN DATA
    # -------------------------------------------------------
    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # -------------------------------------------------------
    # 7. Train Performance
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print("Coefficients:", pipeline.named_steps["model"].coef_)
    print("Intercept:", pipeline.named_steps["model"].intercept_)

    if model_type.lower() == "lassocv":
        print("Best Alpha (LassoCV):", pipeline.named_steps["model"].alpha_)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - train_r2) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 8. Test Performance
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - test_r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 9. Diagnostic Plots
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test Data)")
    plt.show()

    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test Data)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 10. Return Final Model
    # -------------------------------------------------------
    return pipeline

## ElasticNet Regression [Linear]

In [ ]:
def elasticnet_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    model_type="elasticnet",     # "elasticnet" or "elasticnetcv"
    alpha=None,
    l1_ratio=None,
    alphas=None,
    l1_ratios=None,
    cv=5,
    scale=True,
    random_state=42,
    max_iter=5000
):
    """
    ElasticNet / ElasticNetCV Regression with:
    - Combined L1 + L2 regularization
    - k-Fold CV on training data
    - Proper pipeline & diagnostics
    
            Parameters
        ----------
        X_train, y_train : training data
        X_test, y_test   : test data
        model_type : {"elasticnet", "elasticnetcv"}
        alpha : float, optional
            Regularization strength for Lasso
        l1_ratio: float, optional
        alphas : list or array, optional
            Candidate alphas for LassoCV
        l1_ratios: list or array, optional
        cv : int, default=5
            Number of CV folds
        scale : bool, default=True
            Whether to apply StandardScaler
        random_state : int, default=42
        max_iter: 5000

        Returns
        -------
        model : sklearn Pipeline
            Final trained Lasso / LassoCV model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import ElasticNet, ElasticNetCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold, cross_validate
    from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

    # -------------------------------------------------------
    # 1. Handle Defaults
    # -------------------------------------------------------
    if model_type.lower() == "elasticnet":
        if alpha is None:
            alpha = 0.01
        if l1_ratio is None:
            l1_ratio = 0.5
        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=max_iter,
            random_state=random_state
        )

    elif model_type.lower() == "elasticnetcv":
        if alphas is None:
            alphas = np.logspace(-4, 1, 50)
        if l1_ratios is None:
            l1_ratios = [0.1, 0.5, 0.7, 0.9, 1.0]

        model = ElasticNetCV(
            alphas=alphas,
            l1_ratio=l1_ratios,
            cv=cv,
            max_iter=max_iter,
            random_state=random_state
        )

    else:
        raise ValueError("model_type must be 'elasticnet' or 'elasticnetcv'")

    # -------------------------------------------------------
    # 2. Pipeline
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 3. Cross-Validation ONLY on Training Data
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]
    cv_rmse = np.sqrt(cv_mse)

    # -------------------------------------------------------
    # 4. Print CV Results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")

    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. CV Error Distribution Plot
    # -------------------------------------------------------
    plt.figure()
    plt.boxplot(cv_mse)
    plt.title("Cross-Validation MSE Distribution (Train Data)")
    plt.ylabel("MSE")
    plt.show()

    # -------------------------------------------------------
    # 6. Train Final Model on FULL TRAIN DATA
    # -------------------------------------------------------
    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # -------------------------------------------------------
    # 7. Train Performance
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print("Coefficients:", pipeline.named_steps["model"].coef_)
    print("Intercept:", pipeline.named_steps["model"].intercept_)


    if model_type.lower() == "elasticnetcv":
        print("Best Alpha:", pipeline.named_steps["model"].alpha_)
        print("Best L1 Ratio:", pipeline.named_steps["model"].l1_ratio_)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - train_r2) * (len(y_train) - 1) / (len(y_train) - X_train.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 8. Test Performance
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")
    print(
        f"Adjusted R2 : "
        f"{1 - (1 - test_r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1):.4f}"
    )

    # -------------------------------------------------------
    # 9. Diagnostic Plots
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test Data)")
    plt.show()

    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test Data)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 10. Return Final Model
    # -------------------------------------------------------
    return pipeline


## Logistic Regression [Classification - Binary, Multinary]

In [ ]:
def logistic_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,
    random_state=42,
    max_iter=1000,
    multiclass="auto",          # "auto", "ovr", "multinomial"
    class_weight=None,          # None or "balanced"
    scoring="roc_auc",          # default scoring
    search_type=None,           # None, "grid", "random"
    param_grid=None,
    n_iter=20,                   # used only for RandomizedSearchCV
    verbose=1
):
    """
        Comprehensive Logistic Regression Utility with End-to-End Model Evaluation

        This function implements a full machine-learning workflow for Logistic
        Regression, covering preprocessing, model training, validation, hyperparameter
        tuning, evaluation, and diagnostics in a leakage-safe manner.

        Key Features
        ------------
        1. Preprocessing & Pipeline
        - Uses sklearn Pipeline to chain preprocessing and model training
        - Optional feature scaling via StandardScaler
        - Prevents data leakage by fitting preprocessing steps only on training folds

        2. Model Support
        - Logistic Regression for both:
            • Binary classification
            • Multiclass classification (OvR / Multinomial)
        - Supports different solvers, penalties, regularization strengths (C),
            and class weighting strategies

        3. Cross-Validation
        - Uses Stratified K-Fold Cross-Validation (classification-safe)
        - Ensures class distribution is preserved in each fold
        - Performs cross-validation strictly on training data

        4. Hyperparameter Tuning (Optional)
        - Supports both:
            • GridSearchCV (exhaustive search)
            • RandomizedSearchCV (efficient stochastic search)
        - User-selectable search strategy
        - Fully configurable parameter grid/distributions
        - Automatically selects and retrains the best model

        5. Evaluation Metrics
        - Training and testing performance reported separately
        - Metrics include:
            • Accuracy
            • Precision
            • Recall
            • F1-Score
            • ROC-AUC (binary & multiclass OVR)
        - Fold-wise CV metrics and mean CV performance are printed

        6. Model Interpretability
        - Displays learned coefficients and intercepts
        - Helps analyze feature importance and decision influence

        7. Diagnostic Visualizations
        - Cross-validation metric distribution
        - Confusion Matrix
        - ROC Curve (binary classification)
        - Predicted probability distributions

        8. Final Model Output
        - Returns the final trained model:
            • Best estimator from GridSearchCV / RandomizedSearchCV, or
            • Pipeline trained on full training data

        Parameters
        ----------
        X_train : array-like or DataFrame
            Training feature matrix
        y_train : array-like or Series
            Training target labels
        X_test : array-like or DataFrame
            Test feature matrix
        y_test : array-like or Series
            Test target labels
        cv : int
            Number of Stratified K-Fold splits
        scale : bool
            Whether to apply StandardScaler
        multiclass : {'auto', 'ovr', 'multinomial'}
            Strategy for multiclass classification
        class_weight : {None, 'balanced'}
            Class weighting strategy for imbalanced datasets
        scoring : str
            Metric used for model selection during hyperparameter tuning
        search_type : {None, 'grid', 'random'}
            Hyperparameter search strategy
        param_grid : dict or None
            Parameter grid or distributions for tuning
        n_iter : int
            Number of iterations for RandomizedSearchCV
        random_state : int
            Seed for reproducibility
        verbose: int 

        Returns
        -------
        model : sklearn estimator
            Final trained Logistic Regression pipeline or tuned estimator
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import (
        StratifiedKFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        confusion_matrix,
        RocCurveDisplay,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Base Pipeline
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))

    steps.append((
        "model",
        LogisticRegression(
            max_iter=max_iter,
            random_state=random_state,
            multi_class=multiclass,
            class_weight=class_weight
        )
    ))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Stratified K-Fold
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 3. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__penalty": ["l2"],
                "model__C": [0.01, 0.1, 1, 10, 100],
                "model__solver": ["lbfgs"],
                "model__class_weight": [None, "balanced"]
            }

        if search_type == "grid":
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )

        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== HYPERPARAMETER SEARCH RESULTS ==========")
        print("Best Parameters:", search.best_params_)
        print("Best CV Score :", search.best_score_)

    else:
        model = pipeline

        # -------------------------------------------------------
        # 4. Cross-Validation WITHOUT Tuning
        # -------------------------------------------------------
        scoring_dict = {
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted",
            "roc_auc": "roc_auc_ovr"
        }

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring=scoring_dict,
            return_train_score=True
        )

        print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
        for i in range(cv):
            print(
                f"Fold {i+1}: "
                f"Acc={cv_results['test_accuracy'][i]:.4f}, "
                f"Prec={cv_results['test_precision'][i]:.4f}, "
                f"Recall={cv_results['test_recall'][i]:.4f}, "
                f"F1={cv_results['test_f1'][i]:.4f}, "
                f"ROC-AUC={cv_results['test_roc_auc'][i]:.4f}"
            )

        print("\n---------- MEAN CV PERFORMANCE ----------")
        for key in cv_results:
            if key.startswith("test_"):
                print(f"{key}: {cv_results[key].mean():.4f}")

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 5. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    if len(np.unique(y_train)) == 2:
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_train_prob = None
        y_test_prob = None

    # -------------------------------------------------------
    # 6. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")

    if y_train_prob is not None:
        print(f"ROC-AUC  : {roc_auc_score(y_train, y_train_prob):.4f}")
    
    print(classification_report(y_train_pred,y_train))

    # -------------------------------------------------------
    # 7. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")

    if y_test_prob is not None:
        print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_prob):.4f}")

    print(classification_report(y_test_pred,y_test))
    
    # -------------------------------------------------------
    # 8. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # -------------------------------------------------------
    # 9. ROC Curve (Binary Only)
    # -------------------------------------------------------
    if y_test_prob is not None:
        RocCurveDisplay.from_predictions(y_test, y_test_prob)
        plt.title("ROC Curve (Test Data)")
        plt.show()

    return model


## Support Vector Machines [Classification - Linear, Nonlinear]

In [ ]:
def svc_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,
    random_state=42,
    probability=True,
    class_weight=None,
    kernel="rbf",       
    degree=3,            # ✅ for poly
    gamma="scale",       # ✅ for rbf/poly/sigmoid
    coef0=0.0,           # ✅ for poly/sigmoid
    scoring="accuracy",
    search_type=None,           # None, "grid", "random"
    param_grid=None,
    n_iter=30,
    verbose=1
):
    """
    Comprehensive Support Vector Classification (SVC) Utility

    This function provides an end-to-end, leakage-safe implementation of
    Support Vector Machines for classification tasks. It integrates data
    preprocessing, cross-validation, optional hyperparameter tuning,
    model evaluation, and diagnostic visualization into a single reusable
    workflow.

    Key Features
    ------------
    1. Preprocessing & Pipeline
    - Uses sklearn Pipeline to combine preprocessing and model training
    - Optional feature scaling using StandardScaler (strongly recommended
        for SVMs due to margin-based optimization)
    - Prevents data leakage by fitting preprocessing steps only on
        training folds

    2. Model Capabilities
    - Supports binary and multiclass classification problems
    - Enables flexible kernel selection:
        • Linear
        • Polynomial
        • Radial Basis Function (RBF)
        • Sigmoid
    - Allows customization of regularization (C), kernel-specific
        parameters (gamma, degree), and class weighting

    3. Cross-Validation Strategy
    - Uses Stratified K-Fold Cross-Validation to preserve class
        distribution in each fold
    - Ensures reliable and unbiased performance estimation for
        classification tasks
    - Cross-validation is strictly performed on training data

    4. Hyperparameter Optimization (Optional)
    - Supports both:
        • GridSearchCV for exhaustive hyperparameter search
        • RandomizedSearchCV for computationally efficient tuning
    - User-selectable search strategy
    - Automatically identifies and refits the best-performing model

    5. Evaluation Metrics
    - Reports performance separately on training and test datasets
    - Metrics include:
        • Accuracy
        • Precision
        • Recall
        • F1-score
        • ROC-AUC (binary classification when probability=True)
    - Prints fold-wise and mean cross-validation metrics

    6. Diagnostic & Interpretability Tools
    - Confusion matrix visualization for error analysis
    - ROC curve visualization for binary classification
    - Predicted probability analysis (when enabled)
    - Helps diagnose overfitting, underfitting, and class imbalance

    7. Final Model Output
    - Returns the fully trained SVC model:
        • Best estimator from GridSearchCV / RandomizedSearchCV, or
        • Pipeline trained on full training data when tuning is disabled

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training feature matrix
    y_train : array-like or Series
        Training target labels
    X_test : array-like or DataFrame
        Test feature matrix
    y_test : array-like or Series
        Test target labels
    cv : int
        Number of Stratified K-Fold splits
    scale : bool
        Whether to apply feature scaling using StandardScaler
    class_weight : {None, 'balanced'}
        Class weighting strategy for handling imbalanced datasets
    kernel="rbf",       
    degree=3,            # ✅ for poly
    gamma="scale",       # ✅ for rbf/poly/sigmoid
    coef0=0.0,           # ✅ for poly/sigmoid
    probability : bool
        Whether to enable probability estimates (required for ROC-AUC)
    scoring : str
        Metric used for model selection during hyperparameter tuning
    search_type : {None, 'grid', 'random'}
        Hyperparameter search strategy
    param_grid : dict or None
        Parameter grid or distributions for tuning
    n_iter : int
        Number of iterations for RandomizedSearchCV
    random_state : int
        Seed for reproducibility
    verbose: int

    Returns
    -------
    model : sklearn estimator
        Final trained SVC pipeline or tuned estimator
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.svm import SVC
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import (
        StratifiedKFold,
        GridSearchCV,
        RandomizedSearchCV,
        cross_validate
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        confusion_matrix,
        RocCurveDisplay,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Build Pipeline
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))

    if search_type!=None:
        steps.append((
            "model",
            SVC(
                probability=probability,
                class_weight=class_weight,
                random_state=random_state
            )
        ))
    else:
        steps.append((
            "model",
            SVC(
                kernel=kernel,
                degree=degree,
                gamma=gamma,
                coef0=coef0,
                probability=probability,
                class_weight=class_weight,
                random_state=random_state
            )
        ))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Stratified K-Fold
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 3. Auto-fix param_grid (Pipeline Safety)
    # -------------------------------------------------------
    if param_grid is not None:
        fixed_grid = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed_grid[f"model__{k}"] = v
            else:
                fixed_grid[k] = v
        param_grid = fixed_grid

    # -------------------------------------------------------
    # 4. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__kernel": ["rbf", "linear"],
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", "auto"],
                "model__class_weight": [None, "balanced"]
            }

        if search_type == "grid":
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )

        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== HYPERPARAMETER SEARCH RESULTS ==========")
        print("Best Parameters:", search.best_params_)
        print("Best CV Score :", search.best_score_)

    else:
        model = pipeline

        scoring_dict = {
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted"
        }

        if probability and len(np.unique(y_train)) == 2:
            scoring_dict["roc_auc"] = "roc_auc"

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring=scoring_dict
        )

        print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========")
        for i in range(cv):
            metrics = [f"{k}={cv_results[f'test_{k}'][i]:.4f}" for k in scoring_dict]
            print(f"Fold {i+1}: " + ", ".join(metrics))

        print("\n---------- MEAN CV PERFORMANCE ----------")
        for k in scoring_dict:
            print(f"{k}: {cv_results[f'test_{k}'].mean():.4f}")

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 5. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    if probability and len(np.unique(y_train)) == 2:
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_train_prob = None
        y_test_prob = None

    # -------------------------------------------------------
    # 6. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")

    if y_train_prob is not None:
        print(f"ROC-AUC  : {roc_auc_score(y_train, y_train_prob):.4f}")
        
    print(classification_report(y_train_pred,y_train))    

    # -------------------------------------------------------
    # 7. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")

    if y_test_prob is not None:
        print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_prob):.4f}")

    print(classification_report(y_test_pred,y_test))   

    # -------------------------------------------------------
    # 8. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # -------------------------------------------------------
    # 9. ROC Curve (Binary Only)
    # -------------------------------------------------------
    if y_test_prob is not None:
        RocCurveDisplay.from_predictions(y_test, y_test_prob)
        plt.title("ROC Curve (Test Data)")
        plt.show()

    return model


## Suport Vector Machines [Regression]

In [ ]:
def svr_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,
    random_state=42,
    kernel="rbf",
    degree=3,
    gamma="scale",
    C=1.0,
    epsilon=0.1,
    scoring="neg_mean_squared_error",
    search_type=None,        # None | "grid" | "random"
    param_grid=None,
    n_iter=20,
    verbose=1
):
    """
    Support Vector Regression (SVR) with:
    - k-Fold Cross-Validation (TRAIN data only)
    - Proper preprocessing using Pipeline (no data leakage)
    - Optional hyperparameter search (Grid / Random)
    - Train & Test performance evaluation
    - Diagnostic plots
    - Returns final trained model

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    cv : int
        Number of CV folds
    scale : bool
        Whether to apply StandardScaler (RECOMMENDED for SVR)
    kernel : str
        'linear', 'rbf', 'poly'
    degree : int
        Used only if kernel='poly'
    gamma : str or float
        Kernel coefficient
    C : float
        Regularization parameter
    epsilon : float
        Epsilon in epsilon-SVR
    search_type : None | 'grid' | 'random'
        Hyperparameter tuning method
    param_grid : dict
        Hyperparameter grid/distributions
    n_iter : int
        Number of parameter samples (RandomizedSearchCV)
    verbose : int
        Verbosity level

    Returns
    -------
    model : sklearn Pipeline
        Final trained SVR model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.svm import SVR
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import (
        KFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Build base SVR model
    # -------------------------------------------------------
    svr = SVR(
        kernel=kernel,
        degree=degree,
        gamma=gamma,
        C=C,
        epsilon=epsilon
    )

    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", svr))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Optional Hyperparameter Search
    # -------------------------------------------------------

    #  Auto-fix param_grid (Pipeline Safety)

    if param_grid is not None:
        fixed_grid = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed_grid[f"model__{k}"] = v
            else:
                fixed_grid[k] = v
        param_grid = fixed_grid

    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__kernel": ["rbf", "linear"],
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", "auto"]
            }
            
    if search_type == "grid":
        model = GridSearchCV(
            pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            verbose=verbose
        )

    elif search_type == "random":
        model = RandomizedSearchCV(
            pipeline,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            random_state=random_state,
            verbose=verbose
        )
    else:
        model = pipeline

    # -------------------------------------------------------
    # 3. Cross-validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring_dict = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring_dict,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_rmse = np.sqrt(cv_mse)
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]

    # -------------------------------------------------------
    # 4. Print CV results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
    for i in range(cv):
        print(f"Fold {i+1}: "
              f"MSE={cv_mse[i]:.4f}, "
              f"RMSE={cv_rmse[i]:.4f}, "
              f"MAE={cv_mae[i]:.4f}, "
              f"R2={cv_r2[i]:.4f}")

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. Train final model on FULL TRAIN data
    # -------------------------------------------------------
    model.fit(X_train, y_train)

    # If search was used, extract best estimator
    if search_type in ["grid", "random"]:
        print("\nBest Parameters:", model.best_params_)
        final_model = model.best_estimator_
    else:
        final_model = model

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # -------------------------------------------------------
    # 6. Train metrics
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")

    # -------------------------------------------------------
    # 7. Test metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")

    # -------------------------------------------------------
    # 8. Diagnostic plots
    # -------------------------------------------------------
    # Actual vs Predicted (Test)
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test)")
    plt.show()

    # Residuals
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 9. Return final model
    # -------------------------------------------------------
    return final_model


## Naive Bayes Theorem [Classification]

In [ ]:
def naive_bayes_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    model_type="gaussian",   # 'gaussian' | 'multinomial' | 'bernoulli'
    scale=False,
    alpha=1.0,
    binarize=0.0,
    random_state=42
):
    """
    Naive Bayes Classification with:
    - k-Fold Cross-Validation (TRAIN data only)
    - Proper preprocessing using Pipeline (no data leakage)
    - Fold-wise performance reporting
    - Train & Test evaluation
    - Confusion matrix & ROC-AUC (binary)
    - Returns final trained model

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    cv : int
        Number of CV folds
    model_type : str
        'gaussian', 'multinomial', or 'bernoulli'
    scale : bool
        Whether to apply StandardScaler (recommended for GaussianNB)
    alpha : float
        Laplace smoothing parameter (MultinomialNB / BernoulliNB)
    binarize : float
        Threshold for BernoulliNB
    random_state : int
        For reproducibility

    Returns
    -------
    model : sklearn Pipeline
        Final trained Naive Bayes model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold, cross_validate
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        roc_auc_score,
        classification_report
    )

    from sklearn.naive_bayes import (
        GaussianNB,
        MultinomialNB,
        BernoulliNB
    )

    # -------------------------------------------------------
    # 1. Choose Naive Bayes model
    # -------------------------------------------------------
    if model_type == "gaussian":
        nb_model = GaussianNB()
    elif model_type == "multinomial":
        nb_model = MultinomialNB(alpha=alpha)
        scale = False   # MultinomialNB requires non-negative features
    elif model_type == "bernoulli":
        nb_model = BernoulliNB(alpha=alpha, binarize=binarize)
        scale = False
    else:
        raise ValueError("model_type must be 'gaussian', 'multinomial', or 'bernoulli'")

    # -------------------------------------------------------
    # 2. Build Pipeline (prevents data leakage)
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", nb_model))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 3. Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "accuracy": "accuracy",
        "precision": "precision_macro",
        "recall": "recall_macro",
        "f1": "f1_macro"
    }

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=skf,
        scoring=scoring,
        return_train_score=False
    )

    # -------------------------------------------------------
    # 4. Print Fold-wise CV Results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")

    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"Acc={cv_results['test_accuracy'][i]:.4f}, "
            f"Prec={cv_results['test_precision'][i]:.4f}, "
            f"Recall={cv_results['test_recall'][i]:.4f}, "
            f"F1={cv_results['test_f1'][i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean Accuracy : {cv_results['test_accuracy'].mean():.4f}")
    print(f"Mean Precision: {cv_results['test_precision'].mean():.4f}")
    print(f"Mean Recall   : {cv_results['test_recall'].mean():.4f}")
    print(f"Mean F1-score : {cv_results['test_f1'].mean():.4f}")

    # -------------------------------------------------------
    # 5. Train Final Model on FULL TRAIN DATA
    # -------------------------------------------------------
    pipeline.fit(X_train, y_train)

    # Predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # -------------------------------------------------------
    # 6. Train Performance
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print("Accuracy :", accuracy_score(y_train, y_train_pred))
    print("Precision:", precision_score(y_train, y_train_pred, average="macro"))
    print("Recall   :", recall_score(y_train, y_train_pred, average="macro"))
    print("F1-score :", f1_score(y_train, y_train_pred, average="macro"))
    print(classification_report(y_train_pred,y_train))

    # -------------------------------------------------------
    # 7. Test Performance
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print("Accuracy :", accuracy_score(y_test, y_test_pred))
    print("Precision:", precision_score(y_test, y_test_pred, average="macro"))
    print("Recall   :", recall_score(y_test, y_test_pred, average="macro"))
    print("F1-score :", f1_score(y_test, y_test_pred, average="macro"))
    print(classification_report(y_test_pred,y_test))

    # -------------------------------------------------------
    # 8. Confusion Matrix (TEST DATA)
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix (Test Data)")
    plt.show()

    # -------------------------------------------------------
    # 9. ROC-AUC (Binary only)
    # -------------------------------------------------------
    if len(np.unique(y_test)) == 2 and hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_test_prob = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_test_prob)
        print("\nROC-AUC (Test Data):", roc_auc)

    # -------------------------------------------------------
    # 10. Return final model
    # -------------------------------------------------------
    return pipeline


## KNN [Classification]

In [ ]:
def knn_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,
    n_neighbors=5,          # used when search_type=None
    weights="uniform",      # uniform | distance
    metric="minkowski",
    p=2,                    # p=2 → Euclidean, p=1 → Manhattan
    scoring="accuracy",
    search_type=None,       # None | "grid" | "random"
    param_grid=None,
    n_iter=30,
    random_state=42,
    verbose=1
):
    
    """
    Comprehensive K-Nearest Neighbors (KNN) Classification Utility

    This function implements an end-to-end machine learning workflow for
    K-Nearest Neighbors classification, integrating preprocessing, model
    training, validation, hyperparameter tuning, evaluation, and diagnostics
    in a leakage-safe and reusable manner.

    KNN is a distance-based, instance-based (lazy) learning algorithm. This
    function is designed to handle its unique characteristics correctly,
    particularly the strong dependence on feature scaling and distance
    metrics.

    Key Features
    ------------
    1. Preprocessing & Pipeline
    - Uses sklearn Pipeline to chain preprocessing and model training
    - Optional feature scaling using StandardScaler (strongly recommended
        for KNN due to distance-based computations)
    - Prevents data leakage by fitting preprocessing steps exclusively
        on training folds during cross-validation

    2. Model Characteristics
    - Implements KNeighborsClassifier, an instance-based learner with
        no explicit training phase
    - Supports both binary and multiclass classification natively
    - Allows customization of:
        • Number of neighbors (n_neighbors)
        • Neighbor weighting strategy (uniform / distance)
        • Distance metric and Minkowski power (p)

    3. Cross-Validation Strategy
    - Uses Stratified K-Fold Cross-Validation to preserve class
        distributions in each fold
    - Ensures reliable and unbiased performance estimation for
        classification tasks
    - Cross-validation is performed strictly on training data

    4. Hyperparameter Optimization (Optional)
    - Supports:
        • GridSearchCV for exhaustive hyperparameter search
        • RandomizedSearchCV for computationally efficient tuning
    - User-selectable search strategy
    - Automatically refits the best-performing model on the full
        training dataset

    5. Evaluation Metrics
    - Reports model performance separately on training and test datasets
    - Metrics include:
        • Accuracy
        • Precision (weighted)
        • Recall (weighted)
        • F1-score (weighted)
    - Weighted metrics are used to handle class imbalance and multiclass
        settings appropriately
    - Prints fold-wise cross-validation results along with mean scores

    6. Diagnostic & Error Analysis
    - Confusion matrix visualization for analyzing class-wise errors
    - Enables detection of:
        • Overfitting (very low k)
        • Underfitting (very high k)
        • Effects of class imbalance
        • Impact of distance metrics

    7. Final Model Output
    - Returns the final trained KNN model:
        • Best estimator from GridSearchCV / RandomizedSearchCV, or
        • Pipeline trained on the full training data when tuning is disabled
    - Returned model is ready for inference using `.predict()`

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training feature matrix
    y_train : array-like or Series
        Training target labels
    X_test : array-like or DataFrame
        Test feature matrix
    y_test : array-like or Series
        Test target labels
    cv : int
        Number of Stratified K-Fold splits
    scale : bool
        Whether to apply feature scaling using StandardScaler
    n_neighbors : int
        Number of nearest neighbors used for classification
    weights : {'uniform', 'distance'}
        Weighting strategy for neighbor votes
    metric : str
        Distance metric used to compute nearest neighbors
    p : int
        Power parameter for the Minkowski metric
    scoring : str
        Metric used for model selection during hyperparameter tuning
    search_type : {None, 'grid', 'random'}
        Hyperparameter search strategy
    param_grid : dict or None
        Parameter grid or distributions for tuning
    n_iter : int
        Number of iterations for RandomizedSearchCV
    random_state : int
        Seed for reproducibility

    Returns
    -------
    model : sklearn estimator
        Final trained KNN pipeline or tuned estimator
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import (
        StratifiedKFold,
        GridSearchCV,
        RandomizedSearchCV,
        cross_validate
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Build Pipeline
    # -------------------------------------------------------
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))

    steps.append((
        "model",
        KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            metric=metric,
            p=p
        )
    ))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Stratified K-Fold CV
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 3. Auto-fix param_grid for Pipeline
    # -------------------------------------------------------
    if param_grid is not None:
        fixed_grid = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed_grid[f"model__{k}"] = v
            else:
                fixed_grid[k] = v
        param_grid = fixed_grid

    # -------------------------------------------------------
    # 4. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__n_neighbors": [3, 5, 7, 9, 11],
                "model__weights": ["uniform", "distance"],
                "model__metric": ["minkowski"],
                "model__p": [1, 2]
            }

        if search_type == "grid":
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )
        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== HYPERPARAMETER SEARCH RESULTS ==========")
        print("Best Parameters:", search.best_params_)
        print("Best CV Score :", search.best_score_)

    else:
        model = pipeline

        scoring_dict = {
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted"
        }

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring=scoring_dict
        )

        print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========")
        for i in range(cv):
            print(
                f"Fold {i+1}: "
                f"Acc={cv_results['test_accuracy'][i]:.4f}, "
                f"Prec={cv_results['test_precision'][i]:.4f}, "
                f"Recall={cv_results['test_recall'][i]:.4f}, "
                f"F1={cv_results['test_f1'][i]:.4f}"
            )

        print("\n---------- MEAN CV PERFORMANCE ----------")
        for k in scoring_dict:
            print(f"{k}: {cv_results[f'test_{k}'].mean():.4f}")

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 5. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # -------------------------------------------------------
    # 6. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(classification_report(y_train_pred,y_train))

    # -------------------------------------------------------
    # 7. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(classification_report(y_test_pred,y_test))

    # -------------------------------------------------------
    # 8. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return model


## KNN [Regression]

In [ ]:
def knn_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    scale=True,                 # MUST be True for KNN
    n_neighbors=5,
    weights="uniform",          # 'uniform' | 'distance'
    metric="minkowski",
    p=2,                        # p=2 → Euclidean
    scoring="neg_mean_squared_error",
    search_type=None,           # None | "grid" | "random"
    param_grid=None,
    n_iter=20,
    random_state=42,
    verbose=1
):
    """
    K-Nearest Neighbors Regression with:
    - k-Fold Cross-Validation (TRAIN data only)
    - Proper preprocessing using Pipeline (no data leakage)
    - Optional GridSearchCV / RandomizedSearchCV
    - Train & Test performance evaluation
    - Diagnostic plots
    - Returns final trained model

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    cv : int
        Number of CV folds
    scale : bool
        Whether to apply StandardScaler (REQUIRED for KNN)
    n_neighbors : int
        Number of neighbors (k)
    weights : str
        'uniform' or 'distance'
    metric : str
        Distance metric
    p : int
        Power parameter for Minkowski distance
    search_type : None | 'grid' | 'random'
        Hyperparameter tuning method
    param_grid : dict
        Grid / distributions for hyperparameter tuning
    n_iter : int
        Number of parameter samples (RandomizedSearchCV)
    verbose : int
        Verbosity level

    Returns
    -------
    model : sklearn Pipeline
        Final trained KNN regression model
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import (
        KFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Build base KNN model
    # -------------------------------------------------------
    knn = KNeighborsRegressor(
        n_neighbors=n_neighbors,
        weights=weights,
        metric=metric,
        p=p
    )

    if not scale:
        print("⚠️ WARNING: KNN is distance-based — scaling is strongly recommended.")

    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", knn))

    pipeline = Pipeline(steps)

    # -------------------------------------------------------
    # 2. Optional Hyperparameter Search
    # -------------------------------------------------------
    if search_type == "grid":
        model = GridSearchCV(
            pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            verbose=verbose
        )

    elif search_type == "random":
        model = RandomizedSearchCV(
            pipeline,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            random_state=random_state,
            verbose=verbose
        )
    else:
        model = pipeline

    # -------------------------------------------------------
    # 3. Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring_dict = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring_dict,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_rmse = np.sqrt(cv_mse)
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]

    # -------------------------------------------------------
    # 4. Print CV results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. Train final model on FULL TRAIN data
    # -------------------------------------------------------
    model.fit(X_train, y_train)

    if search_type in ["grid", "random"]:
        print("\nBest Parameters:", model.best_params_)
        final_model = model.best_estimator_
    else:
        final_model = model

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # -------------------------------------------------------
    # 6. Train metrics
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")

    # -------------------------------------------------------
    # 7. Test metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")

    # -------------------------------------------------------
    # 8. Diagnostic plots
    # -------------------------------------------------------
    # Actual vs Predicted (Test)
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()]
    )
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test)")
    plt.show()

    # Residuals
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 9. Return final model
    # -------------------------------------------------------
    return final_model


## Decision Tree [Classification]

In [ ]:
def decision_tree_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    random_state=42,
    criterion="gini",           # gini | entropy | log_loss
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    class_weight=None,
    scoring="accuracy",
    search_type=None,           # None | "grid" | "random"
    param_grid=None,
    n_iter=30,
    verbose=1,
    plot_tree_flag=False,         
    max_depth_plot=3,            
    feature_names=None,            
    class_names=None   
):
    """
    Comprehensive Decision Tree Classification Utility

    This function provides an end-to-end machine learning workflow for
    Decision Tree classification, integrating model construction, validation,
    optional hyperparameter tuning, evaluation, and diagnostic analysis in a
    leakage-safe and reusable framework.

    Decision Trees are rule-based, non-parametric models that recursively
    partition the feature space into homogeneous regions. This function is
    designed to exploit their interpretability while mitigating their strong
    tendency to overfit through controlled depth, splitting criteria, and
    cross-validation.

    Key Features
    ------------
    1. Model Construction
    - Implements sklearn's DecisionTreeClassifier
    - Supports multiple splitting criteria:
        • Gini impurity
        • Entropy (information gain)
        • Log-loss
    - Allows explicit control over model complexity using:
        • max_depth
        • min_samples_split
        • min_samples_leaf
        • max_features
    - Supports class weighting to handle imbalanced datasets

    2. Preprocessing Considerations
    - Decision Trees are scale-invariant; feature scaling is not required
    - Operates directly on raw or encoded features
    - Compatible with both numerical and encoded categorical inputs

    3. Cross-Validation Strategy
    - Uses Stratified K-Fold Cross-Validation to preserve class
        distributions in each fold
    - Ensures unbiased and stable performance estimation
    - Cross-validation is performed strictly on training data to
        avoid information leakage

    4. Hyperparameter Optimization (Optional)
    - Supports:
        • GridSearchCV for exhaustive hyperparameter exploration
        • RandomizedSearchCV for computationally efficient tuning
    - User-selectable search strategy
    - Automatically selects and refits the best-performing model on the
        full training dataset

    5. Evaluation Metrics
    - Evaluates performance separately on training and test datasets
    - Metrics include:
        • Accuracy
        • Precision (weighted)
        • Recall (weighted)
        • F1-score (weighted)
    - Weighted metrics ensure correct evaluation in the presence of
        class imbalance or multiclass targets
    - Prints fold-wise cross-validation results along with mean scores

    6. Diagnostic & Error Analysis
    - Generates a confusion matrix for detailed class-wise error analysis
    - Helps identify:
        • Overfitting due to excessive tree depth
        • Underfitting from overly restrictive splits
        • Class imbalance effects
        • Dominant decision paths

    7. Interpretability & Visualization (Optional Extension)
    - Designed to support optional tree visualization using `plot_tree`
    - Enables inspection of:
        • Feature splits and thresholds
        • Node impurity and sample distribution
        • Leaf-level class predictions
    - Facilitates model explainability and debugging

    8. Final Model Output
    - Returns the final trained Decision Tree model:
        • Best estimator from GridSearchCV / RandomizedSearchCV, or
        • Pipeline trained on the full training data when tuning is disabled
    - Returned model is ready for inference using `.predict()` and
        `.predict_proba()`

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training feature matrix
    y_train : array-like or Series
        Training target labels
    X_test : array-like or DataFrame
        Test feature matrix
    y_test : array-like or Series
        Test target labels
    cv : int
        Number of Stratified K-Fold splits
    criterion : {'gini', 'entropy', 'log_loss'}
        Splitting criterion used to measure the quality of a split
    max_depth : int or None
        Maximum depth of the tree (controls overfitting)
    min_samples_split : int
        Minimum number of samples required to split an internal node
    min_samples_leaf : int
        Minimum number of samples required at a leaf node
    max_features : int, float, str or None
        Number of features considered when looking for the best split
    class_weight : dict, 'balanced', or None
        Weights associated with classes for handling imbalance
    scoring : str
        Metric used for model selection during hyperparameter tuning
    search_type : {None, 'grid', 'random'}
        Hyperparameter search strategy
    param_grid : dict or None
        Parameter grid or distributions for tuning
    n_iter : int
        Number of iterations for RandomizedSearchCV
    random_state : int
        Seed for reproducibility
    verbose
    plot_tree_flag=False,         
    max_depth_plot=3,            
    feature_names=None,            
    class_names=None

    Returns
    -------
    model : sklearn estimator
        Final trained Decision Tree pipeline or tuned estimator
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.tree import DecisionTreeClassifier, plot_tree
    from sklearn.pipeline import Pipeline
    from sklearn import tree
    from sklearn.model_selection import (
        StratifiedKFold,
        GridSearchCV,
        RandomizedSearchCV,
        cross_validate
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Build Pipeline (No Scaling Required)
    # -------------------------------------------------------
    pipeline = Pipeline([
        ("model", DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight=class_weight,
            random_state=random_state
        ))
    ])

    # -------------------------------------------------------
    # 2. Stratified K-Fold CV
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 3. Auto-fix param_grid for Pipeline
    # -------------------------------------------------------
    if param_grid is not None:
        fixed_grid = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed_grid[f"model__{k}"] = v
            else:
                fixed_grid[k] = v
        param_grid = fixed_grid

    # -------------------------------------------------------
    # 4. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__criterion": ["gini", "entropy", "log_loss"],
                "model__max_depth": [None, 3, 5, 10, 20],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 5],
                "model__max_features": [None, "sqrt", "log2"],
                "model__class_weight": [None, "balanced"]
            }

        if search_type == "grid":
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )
        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== HYPERPARAMETER SEARCH RESULTS ==========")
        print("Best Parameters:", search.best_params_)
        print("Best CV Score :", search.best_score_)

    else:
        model = pipeline

        scoring_dict = {
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted"
        }

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring=scoring_dict
        )

        print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========")
        for i in range(cv):
            print(
                f"Fold {i+1}: "
                f"Acc={cv_results['test_accuracy'][i]:.4f}, "
                f"Prec={cv_results['test_precision'][i]:.4f}, "
                f"Recall={cv_results['test_recall'][i]:.4f}, "
                f"F1={cv_results['test_f1'][i]:.4f}"
            )

        print("\n---------- MEAN CV PERFORMANCE ----------")
        for k in scoring_dict:
            print(f"{k}: {cv_results[f'test_{k}'].mean():.4f}")

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 5. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # -------------------------------------------------------
    # 6. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(classification_report(y_train_pred,y_train))

    # -------------------------------------------------------
    # 7. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(classification_report(y_test_pred,y_test))

    # -------------------------------------------------------
    # 8. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    
    # -------------------------------------------------------
    # 8. 🌳 TREE VISUALIZATION (NEW)
    # -------------------------------------------------------
    if plot_tree_flag:
        plt.figure(figsize=(20, 10))
        plot_tree(
            model.named_steps["model"],
            feature_names=feature_names,
            class_names=class_names,
            filled=True,
            max_depth=max_depth_plot,
            rounded=True
        )
        plt.title("Decision Tree Visualization")
        plt.show()
        
    return model


## Decision Tree [Regression]

In [ ]:
def decision_tree_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    random_state=42,
    criterion="squared_error",     # 'squared_error' | 'absolute_error'
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    scoring="neg_mean_squared_error",
    search_type=None,              # None | "grid" | "random"
    param_grid=None,
    n_iter=20,
    verbose=1,
    plot_tree_flag=False,         
    max_depth_plot=3,            
    feature_names=None,            
    class_names=None   
):
    """
    Decision Tree Regression with:
    - k-Fold Cross-Validation (TRAIN data only)
    - Fold-wise performance reporting
    - Optional GridSearchCV / RandomizedSearchCV
    - Train & Test evaluation
    - Diagnostic plots
    - Returns final trained model

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    cv : int
        Number of CV folds
    criterion : str
        'squared_error' or 'absolute_error'
    max_depth : int or None
        Maximum tree depth
    min_samples_split : int
        Minimum samples to split a node
    min_samples_leaf : int
        Minimum samples in a leaf
    max_features : int, float or None
        Number of features to consider at split
    search_type : None | 'grid' | 'random'
        Hyperparameter tuning method
    param_grid : dict
        Grid / distributions for hyperparameter tuning
    n_iter : int
        Number of parameter samples (RandomizedSearchCV)
    verbose : int
        Verbosity level
    plot_tree_flag=False,         
    max_depth_plot=3,            
    feature_names=None,            
    class_names=None   

    Returns
    -------
    model : sklearn estimator
        Final trained Decision Tree Regressor
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.tree import DecisionTreeRegressor,plot_tree
    from sklearn.model_selection import (
        KFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Build base Decision Tree model
    # -------------------------------------------------------
    dt = DecisionTreeRegressor(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 2. Optional Hyperparameter Search
    # -------------------------------------------------------
    if search_type == "grid":
        model = GridSearchCV(
            dt,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            verbose=verbose
        )

    elif search_type == "random":
        model = RandomizedSearchCV(
            dt,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            random_state=random_state,
            verbose=verbose
        )
    else:
        model = dt

    # -------------------------------------------------------
    # 3. Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring_dict = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring_dict,
        return_train_score=True
    )

    cv_mse = -cv_results["test_mse"]
    cv_rmse = np.sqrt(cv_mse)
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]

    # -------------------------------------------------------
    # 4. Print CV results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. Train final model on FULL TRAIN data
    # -------------------------------------------------------
    model.fit(X_train, y_train)

    if search_type in ["grid", "random"]:
        print("\nBest Parameters:", model.best_params_)
        final_model = model.best_estimator_
    else:
        final_model = model

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # -------------------------------------------------------
    # 6. Train metrics
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")

    # -------------------------------------------------------
    # 7. Test metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")

    # -------------------------------------------------------
    # 8. Diagnostic plots
    # -------------------------------------------------------
    # Actual vs Predicted (Test)
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()]
    )
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test)")
    plt.show()

    # Residuals
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 8. 🌳 TREE VISUALIZATION (NEW)
    # -------------------------------------------------------
    if plot_tree_flag:
        plt.figure(figsize=(20, 10))
        plot_tree(
            final_model,
            feature_names=feature_names,
            class_names=class_names,
            filled=True,
            max_depth=max_depth_plot,
            rounded=True
        )
        plt.title("Decision Tree Visualization")
        plt.show()
        
    # -------------------------------------------------------
    # 10. Return final model
    # -------------------------------------------------------
    return final_model


## Random Forest [Classification]

In [ ]:
def random_forest_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    random_state=42,
    n_estimators=100,
    criterion="gini",              # gini | entropy | log_loss
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    scoring="accuracy",
    search_type=None,              # None | "grid" | "random"
    param_grid=None,
    n_iter=30,
    plot_feature_importance=True,
    feature_names=None,
    verbose=1
):
    """
    Comprehensive Random Forest Classification Utility

    This function provides an end-to-end machine learning workflow for
    Random Forest classification, integrating ensemble model construction,
    validation, optional hyperparameter tuning, evaluation, and model
    interpretability in a reusable and leakage-safe framework.

    Random Forest is an ensemble learning algorithm that builds multiple
    decision trees using bootstrap sampling (bagging) and random feature
    selection, and aggregates their predictions to improve generalization
    performance and reduce overfitting.

    Key Features
    ------------
    1. Ensemble Model Construction
    - Implements sklearn's RandomForestClassifier
    - Builds an ensemble of decision trees trained on different bootstrap
        samples of the training data
    - Introduces randomness in feature selection at each split to reduce
        correlation between trees
    - Supports multiple splitting criteria:
        • Gini impurity
        • Entropy
        • Log-loss

    2. Preprocessing Considerations
    - Random Forests are tree-based and scale-invariant
    - Feature scaling is not required and therefore not applied
    - Works directly with numerical features and encoded categorical data

    3. Cross-Validation Strategy
    - Uses Stratified K-Fold Cross-Validation to preserve class
        distributions across folds
    - Ensures stable and unbiased performance estimation for both binary
        and multiclass classification tasks
    - Cross-validation is strictly applied only to training data to avoid
        information leakage

    4. Hyperparameter Optimization (Optional)
    - Supports:
        • GridSearchCV for exhaustive hyperparameter tuning
        • RandomizedSearchCV for efficient exploration of large parameter
            spaces
    - Allows tuning of key ensemble parameters such as:
        • Number of trees (n_estimators)
        • Tree depth (max_depth)
        • Minimum samples per split/leaf
        • Feature subsampling strategy (max_features)
        • Class weighting for imbalanced datasets
    - Automatically refits the best-performing model on the full training
        dataset

    5. Evaluation Metrics
    - Evaluates model performance separately on training and test datasets
    - Metrics include:
        • Accuracy
        • Precision (weighted)
        • Recall (weighted)
        • F1-score (weighted)
    - Weighted metrics ensure robust evaluation in multiclass and
        imbalanced classification settings
    - Prints fold-wise and mean cross-validation performance metrics

    6. Diagnostic & Error Analysis
    - Generates a confusion matrix for detailed class-wise error analysis
    - Helps detect:
        • Overfitting or underfitting
        • Class imbalance effects
        • Misclassification patterns across classes

    7. Model Interpretability
    - Computes and visualizes feature importance scores based on impurity
        reduction across the ensemble
    - Enables identification of influential features driving model
        predictions
    - Facilitates model debugging and feature selection insights

    8. Final Model Output
    - Returns the final trained Random Forest model:
        • Best estimator from GridSearchCV / RandomizedSearchCV, or
        • Pipeline trained on the full training data when tuning is
            disabled
    - Returned model is ready for inference using `.predict()` and
        `.predict_proba()`

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training feature matrix
    y_train : array-like or Series
        Training target labels
    X_test : array-like or DataFrame
        Test feature matrix
    y_test : array-like or Series
        Test target labels
    cv : int
        Number of Stratified K-Fold splits
    n_estimators : int
        Number of trees in the forest
    criterion : {'gini', 'entropy', 'log_loss'}
        Function to measure the quality of a split
    max_depth : int or None
        Maximum depth of each decision tree
    min_samples_split : int
        Minimum number of samples required to split an internal node
    min_samples_leaf : int
        Minimum number of samples required at a leaf node
    max_features : {'sqrt', 'log2', None}
        Number of features considered when looking for the best split
    bootstrap : bool
        Whether bootstrap samples are used when building trees
    class_weight : dict, 'balanced', or None
        Class weights to handle imbalanced datasets
    scoring : str
        Metric used for model selection during hyperparameter tuning
    search_type : {None, 'grid', 'random'}
        Hyperparameter search strategy
    param_grid : dict or None
        Parameter grid or distributions for tuning
    n_iter : int
        Number of iterations for RandomizedSearchCV
    random_state : int
        Seed for reproducibility

    Returns
    -------
    model : sklearn estimator
        Final trained Random Forest pipeline or tuned estimator
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import (
        StratifiedKFold,
        GridSearchCV,
        RandomizedSearchCV,
        cross_validate
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Build Pipeline (No Scaling Needed)
    # -------------------------------------------------------
    pipeline = Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=n_estimators,
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            class_weight=class_weight,
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    # -------------------------------------------------------
    # 2. Stratified K-Fold CV
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 3. Auto-fix param_grid for Pipeline
    # -------------------------------------------------------
    if param_grid is not None:
        fixed_grid = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed_grid[f"model__{k}"] = v
            else:
                fixed_grid[k] = v
        param_grid = fixed_grid

    # -------------------------------------------------------
    # 4. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if param_grid is None:
            param_grid = {
                "model__n_estimators": [100, 200, 500],
                "model__max_depth": [None, 5, 10, 20],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 5],
                "model__max_features": ["sqrt", "log2"],
                "model__class_weight": [None, "balanced"]
            }

        if search_type == "grid":
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )
        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== HYPERPARAMETER SEARCH RESULTS ==========")
        print("Best Parameters:", search.best_params_)
        print("Best CV Score :", search.best_score_)

    else:
        model = pipeline

        scoring_dict = {
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted"
        }

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring=scoring_dict
        )

        print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========")
        for i in range(cv):
            print(
                f"Fold {i+1}: "
                f"Acc={cv_results['test_accuracy'][i]:.4f}, "
                f"Prec={cv_results['test_precision'][i]:.4f}, "
                f"Recall={cv_results['test_recall'][i]:.4f}, "
                f"F1={cv_results['test_f1'][i]:.4f}"
            )

        print("\n---------- MEAN CV PERFORMANCE ----------")
        for k in scoring_dict:
            print(f"{k}: {cv_results[f'test_{k}'].mean():.4f}")

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 5. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # -------------------------------------------------------
    # 6. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(classification_report(y_train_pred,y_train))
    

    # -------------------------------------------------------
    # 7. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(classification_report(y_test_pred,y_test))
    # -------------------------------------------------------
    # 8. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # -------------------------------------------------------
    # 9. Feature Importance (Optional)
    # -------------------------------------------------------
    if plot_feature_importance:
        importances = model.named_steps["model"].feature_importances_
        indices = np.argsort(importances)[::-1]

        if feature_names is None:
            feature_names = [f"X{i}" for i in range(len(importances))]

        plt.figure(figsize=(10, 6))
        plt.bar(range(len(importances)), importances[indices])
        plt.xticks(range(len(importances)),
                   np.array(feature_names)[indices],
                   rotation=90)
        plt.title("Feature Importance (Random Forest)")
        plt.tight_layout()
        plt.show()
    
    from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

    # -------------------------------------------------------
    # ROC Curve (Binary Classification Only)
    # -------------------------------------------------------
    
    if len(np.unique(y_test)) == 2:
        y_test_prob = model.named_steps["model"].predict_proba(X_test)[:, 1]

        auc_score = roc_auc_score(y_test, y_test_prob)

        RocCurveDisplay.from_predictions(
            y_test,
            y_test_prob,
            name=f"Random Forest (AUC = {auc_score:.4f})"
        )

        plt.title("ROC Curve (Test Data)")
        plt.grid(True)
        plt.show()


    return model

## Random Forest [Regression]

In [ ]:
def random_forest_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5,
    random_state=42,
    n_estimators=100,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    scoring="neg_mean_squared_error",
    search_type=None,              # None | "grid" | "random"
    param_grid=None,
    n_iter=20,
    verbose=1
):
    """
    Random Forest Regression with:
    - k-Fold Cross-Validation (TRAIN data only)
    - Fold-wise performance reporting
    - Optional GridSearchCV / RandomizedSearchCV
    - Train & Test evaluation
    - Diagnostic plots
    - Returns final trained model

    Parameters
    ----------
    X_train, y_train, X_test, y_test : array-like
        Train/Test data
    cv : int
        Number of CV folds
    n_estimators : int
        Number of trees
    criterion : str
        'squared_error' or 'absolute_error'
    max_depth : int or None
        Maximum depth of trees
    min_samples_split : int
        Minimum samples to split
    min_samples_leaf : int
        Minimum samples per leaf
    max_features : str, int, float
        Number of features to consider at split
    bootstrap : bool
        Whether bootstrap samples are used
    search_type : None | 'grid' | 'random'
        Hyperparameter tuning method
    param_grid : dict
        Grid / distributions for hyperparameter tuning
    n_iter : int
        Number of parameter samples (RandomizedSearchCV)
    verbose : int
        Verbosity level

    Returns
    -------
    model : sklearn estimator
        Final trained Random Forest Regressor
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import (
        KFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    # -------------------------------------------------------
    # 1. Build base Random Forest model
    # -------------------------------------------------------
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
        n_jobs=-1
    )

    # -------------------------------------------------------
    # 2. Optional Hyperparameter Search
    # -------------------------------------------------------
    if search_type == "grid":
        model = GridSearchCV(
            rf,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            verbose=verbose,
            n_jobs=-1
        )

    elif search_type == "random":
        model = RandomizedSearchCV(
            rf,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            random_state=random_state,
            verbose=verbose,
            n_jobs=-1
        )
    else:
        model = rf

    # -------------------------------------------------------
    # 3. Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring_dict = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring_dict,
        return_train_score=True,
        n_jobs=-1
    )

    cv_mse = -cv_results["test_mse"]
    cv_rmse = np.sqrt(cv_mse)
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]

    # -------------------------------------------------------
    # 4. Print CV results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. Train final model on FULL TRAIN data
    # -------------------------------------------------------
    model.fit(X_train, y_train)

    if search_type in ["grid", "random"]:
        print("\nBest Parameters:", model.best_params_)
        final_model = model.best_estimator_
    else:
        final_model = model

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # -------------------------------------------------------
    # 6. Train metrics
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    print(f"MSE  : {train_mse:.4f}")
    print(f"RMSE : {train_rmse:.4f}")
    print(f"MAE  : {train_mae:.4f}")
    print(f"R2   : {train_r2:.4f}")

    # -------------------------------------------------------
    # 7. Test metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"MSE  : {test_mse:.4f}")
    print(f"RMSE : {test_rmse:.4f}")
    print(f"MAE  : {test_mae:.4f}")
    print(f"R2   : {test_r2:.4f}")

    # -------------------------------------------------------
    # 8. Diagnostic plots
    # -------------------------------------------------------
    # Actual vs Predicted (Test)
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()]
    )
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test)")
    plt.show()

    # Residuals
    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 9. Return final model
    # -------------------------------------------------------
    return final_model


## AdaBoost, GradientBoost, XgBoost [Classification]

In [ ]:
def boosting_classification(
    X_train,
    y_train,
    X_test,
    y_test,
    model_type="adaboost",        # "adaboost" | "gradient_boost" | "xgboost"
    cv=5,
    random_state=42,
    scoring="accuracy",
    search_type=None,             # None | "grid" | "random"
    param_grid=None,
    n_iter=30,
    verbose=1
):
    """
    Unified Boosting Classification Utility

    This function provides a unified, end-to-end machine learning workflow
    for tree-based boosting algorithms used in classification tasks. It
    allows seamless selection between multiple boosting strategies while
    maintaining a consistent and leakage-safe modeling pipeline.

    Supported Algorithms
    --------------------
    1. AdaBoost (Adaptive Boosting)
    - Sequentially trains weak learners (decision stumps by default)
    - Increases focus on misclassified samples in subsequent iterations
    - Effective for reducing bias on moderately complex datasets

    2. Gradient Boosting
    - Builds trees sequentially by optimizing a differentiable loss
        function using gradient descent
    - Captures complex non-linear decision boundaries
    - Offers fine-grained control over bias–variance tradeoff via learning
        rate and tree depth

    3. XGBoost (Extreme Gradient Boosting)
    - Highly optimized and regularized implementation of gradient
        boosting
    - Incorporates shrinkage, column subsampling, and tree pruning
    - Scales efficiently to large datasets and high-dimensional feature
        spaces
    - Widely used in competitive machine learning and production systems

    Key Features
    ------------
    1. Unified Model Selection
    - Single interface to switch between AdaBoost, Gradient Boosting, and
        XGBoost using a simple `model_type` parameter
    - Ensures consistent preprocessing, validation, and evaluation logic
        across different boosting algorithms

    2. Preprocessing Considerations
    - Boosting algorithms are tree-based and scale-invariant
    - Feature scaling is not required and therefore not applied
    - Operates directly on numerical and encoded categorical features

    3. Cross-Validation Strategy
    - Uses Stratified K-Fold Cross-Validation to preserve class
        distributions in each fold
    - Ensures stable and unbiased performance estimation for both binary
        and multiclass classification tasks
    - Cross-validation is performed strictly on training data to avoid
        information leakage

    4. Hyperparameter Optimization (Optional)
    - Supports:
        • GridSearchCV for exhaustive hyperparameter tuning
        • RandomizedSearchCV for efficient exploration of large parameter
            spaces
    - Allows tuning of critical boosting parameters such as:
        • Number of estimators
        • Learning rate
        • Tree depth and subsampling ratios
        • Feature subsampling (for XGBoost)
    - Automatically selects and refits the best-performing model on the
        full training dataset

    5. Evaluation Metrics
    - Evaluates model performance separately on training and test datasets
    - Metrics include:
        • Accuracy
        • Precision (weighted)
        • Recall (weighted)
        • F1-score (weighted)
        • ROC-AUC (binary classification only)
    - Weighted metrics ensure robust evaluation in the presence of class
        imbalance and multiclass targets

    6. Diagnostic & Error Analysis
    - Generates a confusion matrix for detailed class-wise error analysis
    - Supports ROC curve visualization for binary classification problems
    - Helps identify:
        • Overfitting or underfitting
        • Class imbalance effects
        • Misclassification patterns

    7. Model Interpretability
    - Provides access to feature importance scores (model-dependent)
    - Facilitates understanding of which features contribute most to
        predictions
    - Supports model comparison across different boosting techniques

    8. Final Model Output
    - Returns the final trained boosting classifier:
        • Best estimator from GridSearchCV / RandomizedSearchCV, or
        • Pipeline trained on the full training data when tuning is
            disabled
    - Returned model is ready for inference using `.predict()` and
        `.predict_proba()`

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training feature matrix
    y_train : array-like or Series
        Training target labels
    X_test : array-like or DataFrame
        Test feature matrix
    y_test : array-like or Series
        Test target labels
    model_type : {'adaboost', 'gradient_boost', 'xgboost'}
        Boosting algorithm to use
    cv : int
        Number of Stratified K-Fold splits
    scoring : str
        Metric used for model selection during hyperparameter tuning
    search_type : {None, 'grid', 'random'}
        Hyperparameter search strategy
    param_grid : dict or None
        Parameter grid or distributions for tuning
    n_iter : int
        Number of iterations for RandomizedSearchCV
    random_state : int
        Seed for reproducibility

    Returns
    -------
    model : sklearn estimator
        Final trained boosting classifier
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import (
        StratifiedKFold,
        GridSearchCV,
        RandomizedSearchCV,
        cross_validate
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        roc_auc_score,
        RocCurveDisplay,
        classification_report
    )

    # -------------------------------------------------------
    # 1. Select Model
    # -------------------------------------------------------
    if model_type == "adaboost":
        from sklearn.ensemble import AdaBoostClassifier

        model = AdaBoostClassifier(
            n_estimators=100,
            learning_rate=1.0,
            random_state=random_state
        )

        default_grid = {
            "model__n_estimators": [50, 100, 200],
            "model__learning_rate": [0.01, 0.1, 1.0]
        }

    elif model_type == "gradient_boost":
        from sklearn.ensemble import GradientBoostingClassifier

        model = GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=random_state
        )

        default_grid = {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.01, 0.1],
            "model__max_depth": [3, 5]
        }

    elif model_type == "xgboost":
        from xgboost import XGBClassifier

        model = XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            use_label_encoder=False,
            random_state=random_state
        )

        default_grid = {
            "model__n_estimators": [100, 300],
            "model__learning_rate": [0.01, 0.1],
            "model__max_depth": [3, 6],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }

    else:
        raise ValueError("model_type must be 'adaboost', 'gradient_boost', or 'xgboost'")

    # -------------------------------------------------------
    # 2. Pipeline
    # -------------------------------------------------------
    pipeline = Pipeline([
        ("model", model)
    ])

    # -------------------------------------------------------
    # 3. Stratified K-Fold
    # -------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    # -------------------------------------------------------
    # 4. Fix param_grid for Pipeline
    # -------------------------------------------------------
    if param_grid is not None:
        fixed = {}
        for k, v in param_grid.items():
            if not k.startswith("model__"):
                fixed[f"model__{k}"] = v
            else:
                fixed[k] = v
        param_grid = fixed
    else:
        param_grid = default_grid

    # -------------------------------------------------------
    # 5. Hyperparameter Search (Optional)
    # -------------------------------------------------------
    if search_type is not None:

        if search_type == "grid":
            search = GridSearchCV(
                pipeline,
                param_grid,
                scoring=scoring,
                cv=skf,
                n_jobs=-1,
                verbose=verbose
            )

        elif search_type == "random":
            search = RandomizedSearchCV(
                pipeline,
                param_grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=skf,
                random_state=random_state,
                n_jobs=-1,
                verbose=verbose
            )

        else:
            raise ValueError("search_type must be None, 'grid', or 'random'")

        search.fit(X_train, y_train)
        model = search.best_estimator_

        print("\n========== BEST PARAMETERS ==========")
        print(search.best_params_)
        print("Best CV Score:", search.best_score_)

    else:
        model = pipeline

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=skf,
            scoring={
                "accuracy": "accuracy",
                "precision": "precision_weighted",
                "recall": "recall_weighted",
                "f1": "f1_weighted"
            }
        )

        print("\n========== CROSS VALIDATION ==========")
        for i in range(cv):
            print(
                f"Fold {i+1}: "
                f"Acc={cv_results['test_accuracy'][i]:.4f}, "
                f"Prec={cv_results['test_precision'][i]:.4f}, "
                f"Recall={cv_results['test_recall'][i]:.4f}, "
                f"F1={cv_results['test_f1'][i]:.4f}"
            )

        model.fit(X_train, y_train)

    # -------------------------------------------------------
    # 6. Predictions
    # -------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # -------------------------------------------------------
    # 7. TRAIN PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_train, y_train_pred, average='weighted'):.4f}")
    print(classification_report(y_train_pred,y_train))
    

    # -------------------------------------------------------
    # 8. TEST PERFORMANCE
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
    print(classification_report(y_test_pred,y_test))

    # -------------------------------------------------------
    # 9. Confusion Matrix
    # -------------------------------------------------------
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test Data)")
    plt.show()

    # -------------------------------------------------------
    # 10. ROC Curve (Binary Only)
    # -------------------------------------------------------
    if len(np.unique(y_test)) == 2:
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        RocCurveDisplay.from_predictions(
            y_test, y_prob,
            name=f"{model_type.upper()} (AUC={auc:.4f})"
        )
        plt.show()

    return model


## AdaBoost, GradientBoost, XgBoost [Regression]

In [ ]:
def boosting_regression(
    X_train,
    y_train,
    X_test,
    y_test,
    model_type="adaboost",        # 'adaboost' | 'gboost' | 'xgboost'
    cv=5,
    random_state=42,

    # Common parameters
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,

    # AdaBoost specific
    loss="linear",

    # Gradient Boosting specific
    subsample=1.0,

    # XGBoost specific
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,

    scoring="neg_mean_squared_error",
    search_type=None,             # None | "grid" | "random"
    param_grid=None,
    n_iter=20,
    verbose=1
):
    """
    Train and evaluate boosting-based regression models (AdaBoost, Gradient Boosting,
    or XGBoost) using a leakage-safe machine learning workflow.

    The function supports k-fold cross-validation performed only on the training
    data, optional hyperparameter tuning via GridSearchCV or RandomizedSearchCV,
    and comprehensive evaluation on both training and test datasets. Diagnostic
    plots are generated to help analyze model fit and residual behavior.

    Tree-based boosting models do not require feature scaling, as splits are based
    on feature thresholds rather than distance calculations.

    Parameters
    ----------
    X_train : array-like of shape (n_samples, n_features)
        Training feature matrix.

    y_train : array-like of shape (n_samples,)
        Training target values.

    X_test : array-like of shape (n_samples, n_features)
        Test feature matrix.

    y_test : array-like of shape (n_samples,)
        Test target values.

    model_type : {'adaboost', 'gboost', 'xgboost'}, default='adaboost'
        Type of boosting regression model to use.

    cv : int, default=5
        Number of folds for k-fold cross-validation.

    random_state : int, default=42
        Controls reproducibility of results.

    n_estimators : int, default=100
        Number of boosting stages to perform.

    learning_rate : float, default=0.1
        Shrinks the contribution of each tree.

    max_depth : int, default=3
        Maximum depth of individual regression trees.

    loss : str, default='linear'
        Loss function used in AdaBoost regression.

    subsample : float, default=1.0
        Fraction of samples used for fitting individual base learners
        (used in Gradient Boosting and XGBoost).

    colsample_bytree : float, default=1.0
        Fraction of features used for each tree (XGBoost only).

    reg_alpha : float, default=0.0
        L1 regularization term on weights (XGBoost only).

    reg_lambda : float, default=1.0
        L2 regularization term on weights (XGBoost only).

    scoring : str, default='neg_mean_squared_error'
        Metric used for cross-validation and hyperparameter tuning.

    search_type : {None, 'grid', 'random'}, default=None
        Type of hyperparameter search to perform.

    param_grid : dict or None, default=None
        Hyperparameter grid or distributions for search.

    n_iter : int, default=20
        Number of parameter settings sampled in RandomizedSearchCV.

    verbose : int, default=1
        Verbosity level for hyperparameter search.

    Returns
    -------
    model : estimator
        Final trained boosting regression model, fitted on the full training data.
    """


    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.model_selection import (
        KFold,
        cross_validate,
        GridSearchCV,
        RandomizedSearchCV
    )
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        r2_score
    )

    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor

    # -------------------------------------------------------
    # 1. Choose Boosting Model
    # -------------------------------------------------------
    if model_type == "adaboost":
        base_estimator = DecisionTreeRegressor(
            max_depth=max_depth,
            random_state=random_state
        )

        model = AdaBoostRegressor(
            estimator=base_estimator,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            loss=loss,
            random_state=random_state
        )

    elif model_type == "gboost":
        model = GradientBoostingRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            random_state=random_state
        )

    elif model_type == "xgboost":
        try:
            from xgboost import XGBRegressor
        except ImportError:
            raise ImportError("XGBoost is not installed. Install it using: pip install xgboost")

        model = XGBRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0
        )

    else:
        raise ValueError("model_type must be 'adaboost', 'gboost', or 'xgboost'")

    # -------------------------------------------------------
    # 2. Optional Hyperparameter Search
    # -------------------------------------------------------
    if search_type == "grid":
        model = GridSearchCV(
            model,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            verbose=verbose,
            n_jobs=-1
        )

    elif search_type == "random":
        model = RandomizedSearchCV(
            model,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            random_state=random_state,
            verbose=verbose,
            n_jobs=-1
        )

    # -------------------------------------------------------
    # 3. Cross-Validation (TRAIN DATA ONLY)
    # -------------------------------------------------------
    kf = KFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    scoring_dict = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring=scoring_dict,
        return_train_score=True,
        n_jobs=-1
    )

    cv_mse = -cv_results["test_mse"]
    cv_rmse = np.sqrt(cv_mse)
    cv_mae = -cv_results["test_mae"]
    cv_r2 = cv_results["test_r2"]

    # -------------------------------------------------------
    # 4. Print CV results
    # -------------------------------------------------------
    print("\n========== CROSS VALIDATION (TRAIN DATA ONLY) ==========\n")
    for i in range(cv):
        print(
            f"Fold {i+1}: "
            f"MSE={cv_mse[i]:.4f}, "
            f"RMSE={cv_rmse[i]:.4f}, "
            f"MAE={cv_mae[i]:.4f}, "
            f"R2={cv_r2[i]:.4f}"
        )

    print("\n---------- MEAN CV PERFORMANCE ----------")
    print(f"Mean MSE  : {cv_mse.mean():.4f}")
    print(f"Mean RMSE : {cv_rmse.mean():.4f}")
    print(f"Mean MAE  : {cv_mae.mean():.4f}")
    print(f"Mean R2   : {cv_r2.mean():.4f}")

    # -------------------------------------------------------
    # 5. Train Final Model on FULL TRAIN data
    # -------------------------------------------------------
    model.fit(X_train, y_train)

    if search_type in ["grid", "random"]:
        print("\nBest Parameters:", model.best_params_)
        final_model = model.best_estimator_
    else:
        final_model = model

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # -------------------------------------------------------
    # 6. Train Metrics
    # -------------------------------------------------------
    print("\n========== TRAIN DATA PERFORMANCE ==========")
    print("MSE :", mean_squared_error(y_train, y_train_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))
    print("MAE :", mean_absolute_error(y_train, y_train_pred))
    print("R2  :", r2_score(y_train, y_train_pred))

    # -------------------------------------------------------
    # 7. Test Metrics
    # -------------------------------------------------------
    print("\n========== TEST DATA PERFORMANCE ==========")
    print("MSE :", mean_squared_error(y_test, y_test_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))
    print("MAE :", mean_absolute_error(y_test, y_test_pred))
    print("R2  :", r2_score(y_test, y_test_pred))

    # -------------------------------------------------------
    # 8. Diagnostic Plots
    # -------------------------------------------------------
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()]
    )
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted (Test)")
    plt.show()

    residuals = y_test - y_test_pred

    plt.figure()
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Predicted (Test)")
    plt.show()

    sns.displot(residuals, kind="kde")

    # -------------------------------------------------------
    # 9. Return final model
    # -------------------------------------------------------
    return final_model
